# Stage I — affine Boy colormap optimized in OKLab

The displayed map remains
$$c_{\rm sRGB}(n)=A p_B(n)+b,$$
with the hard global encoded-sRGB gamut condition $0\le c_k(n)\le1$. Perceptual utilities are evaluated after converting the displayed sRGB color to OKLab. The axis semantics remain $e_x\to$ red, $e_y\to$ green, $e_z\to$ blue; the three targets are the OKLab coordinates of exact sRGB `(1,0,0)`, `(0,1,0)`, `(0,0,1)`.

The Pareto problem is
$$\min J_{\rm axis}^{\rm OKLab}\quad\text{s.t.}\quad J_{\rm loc}^{\rm OKLab}\le t,\quad c_{\rm sRGB}(n)\in[0,1]^3.$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
np.set_printoptions(precision=10, suppress=True)

def boy_map(n):
    n=np.asarray(n,float); x,y,z=np.moveaxis(n,-1,0); x2,y2,z2=x*x,y*y,z*z
    p1=.5*((2*x2-y2-z2)+2*y*z*(y2-z2)+z*x*(x2-z2)+x*y*(y2-x2))
    p2=.875*((y2-z2)+z*x*(z2-x2)+x*y*(y2-x2))
    p3=.125*(x+y+z)*((x+y+z)**3+4*(y-x)*(z-y)*(x-z))
    return np.stack((p1,p2,p3),axis=-1)

def fibonacci_sphere(n):
    i=np.arange(n,dtype=float); z=1-2*(i+.5)/n; phi=np.pi*(3-np.sqrt(5))*i
    r=np.sqrt(np.maximum(0,1-z*z)); return np.c_[r*np.cos(phi),r*np.sin(phi),z]

P_AXIS=boy_map(np.eye(3))
def pack(A,b): return np.r_[np.asarray(A).ravel(),np.asarray(b).ravel()]
def unpack(v): return v[:9].reshape(3,3),v[9:12]
def axis_colors(A,b): return P_AXIS@A.T+b

## OKLab objectives

$$J_{\rm axis}^{\rm OKLab}=\sum_{\alpha=x,y,z}\|\operatorname{OKLab}(c(e_\alpha))-\operatorname{OKLab}(e_\alpha)\|^2.$$
For local uniformity, with $f(n)=\operatorname{OKLab}(c(n))$ and tangent derivative $D_Tf$, let $G=(D_Tf)^T(D_Tf)$. Then
$$J_{\rm loc}^{\rm OKLab}=\frac{\langle\operatorname{tr}(G^2)\rangle}{\langle\operatorname{tr}G\rangle^2}-\frac12.$$

In [ ]:
M1=np.array([[.4122214708,.5363325363,.0514459929],[.2119034982,.6806995451,.1073969566],[.0883024619,.2817188376,.6299787005]])
M2=np.array([[.2104542553,.793617785,-.0040720468],[1.9779984951,-2.428592205,.4505937099],[.0259040371,.7827717662,-.808675766]])

def srgb_to_linear(c):
    c=np.asarray(c,float); out=np.empty_like(c); m=c<=.04045
    out[m]=c[m]/12.92; out[~m]=((c[~m]+.055)/1.055)**2.4; return out

def srgb_to_oklab(c): return np.cbrt(srgb_to_linear(c)@M1.T)@M2.T
TARGET=srgb_to_oklab(np.eye(3))

def J_axis(v):
    A,b=unpack(v); lab=srgb_to_oklab(np.clip(axis_colors(A,b),0,1))
    return np.sum((lab-TARGET)**2)

def dlinear_dsrgb(c):
    out=np.empty_like(c); m=c<=.04045; out[m]=1/12.92
    out[~m]=(2.4/1.055)*((c[~m]+.055)/1.055)**1.4; return out

N_LOCAL=1200
N_DIR=fibonacci_sphere(N_LOCAL); P_LOCAL=boy_map(N_DIR)
PROJ=np.eye(3)[None]-N_DIR[:,:,None]*N_DIR[:,None,:]
_eps=2e-6; B=np.empty((N_LOCAL,3,3))
for j in range(3):
    d=np.zeros(3); d[j]=_eps
    B[:,:,j]=(boy_map(N_DIR+d)-boy_map(N_DIR-d))/(2*_eps)

def oklab_jacobian_srgb(c):
    lin=srgb_to_linear(c); lms=lin@M1.T
    dc=(1/3)*np.maximum(lms,1e-12)**(-2/3); dl=dlinear_dsrgb(c)
    return np.einsum('ab,nb,bc,nc->nac',M2,dc,M1,dl)

def J_local(v):
    A,b=unpack(v); c=np.clip(P_LOCAL@A.T+b,0,1); Jc=oklab_jacobian_srgb(c)
    D=np.einsum('nij,jk,nkl->nil',Jc,A,B); DT=np.einsum('nij,njk->nik',D,PROJ)
    G=np.einsum('nji,njk->nik',DT,DT); tr=np.trace(G,axis1=1,axis2=2); tr2=np.einsum('nij,nji->n',G,G)
    m=tr.mean(); return tr2.mean()/(m*m)-.5

## Computed Pareto frontier

A reference run used 1200 deterministic Fibonacci directions for the OKLab local quadrature, 3500 initial gamut constraints, 100000-direction gamut verification with constraint generation, and SLSQP continuation. The resulting frontier was:

| $J_{loc}^{OKLab}$ | $J_{axis}^{OKLab}$ |
|---:|---:|
|0.760|0.00218045|
|0.700|0.00256937|
|0.600|0.00379839|
|0.500|0.01000066|
|0.460|0.01489886|
|0.440|0.01807508|
|0.430|0.01988217|
|0.420|0.02185428|
|0.410|0.02401155|
|0.400|0.02637681|
|0.380|0.03186053|
|0.350|0.04289584|
|0.300|0.08609069|

After normalizing both objective axes over this scanned range, the point of maximum perpendicular distance from the endpoint chord is $J_{loc}^{OKLab}=0.43$, so this is used as the current knee point.

In [ ]:
pareto_table=pd.DataFrame({
    'J_local_OKLab':[.76,.70,.60,.50,.46,.44,.43,.42,.41,.40,.38,.35,.30],
    'J_axis_OKLab':[.0021804532,.0025693697,.0037983884,.0100006569,.0148988625,.0180750782,.0198821690,.0218542850,.0240115472,.0263768063,.0318605325,.0428958357,.0860906875],
})
display(pareto_table)
plt.figure(figsize=(6,4))
plt.plot(pareto_table.J_local_OKLab,pareto_table.J_axis_OKLab,'o-')
plt.scatter([.43],[.0198821690],s=80,label='selected knee')
plt.xlabel(r'$J_{loc}^{OKLab}$'); plt.ylabel(r'$J_{axis}^{OKLab}$'); plt.legend(); plt.grid(alpha=.25); plt.show()

## Selected gamut-safe map

The raw $t=0.43$ numerical solution touched the sampled gamut boundary. A dense $10^6$-direction verification plus local continuous extremum refinement found red/blue excursions of only a few $10^{-5}$ outside the cube. Before export, only the red and blue rows were contracted by less than $4.4\times10^{-5}$ and shifted by about $10^{-5}$; this changes the objectives negligibly while putting the continuous extrema strictly inside the strict Nematics3D RGB validator.

The exported map is
$$c(n)=A p_B(n)+b$$
with the matrix and offset below. Its axis colors remain close to the intended red/green/blue primaries.

In [ ]:
A_SELECTED=np.array([[ 0.5015275525743265, 0.0814208604228778, 0.4289041134674454],[-0.2426021621724047, 0.3331598986797062, 0.2349957727825471],[-0.2761140886939694,-0.3089204069097675, 0.4083459067954693]])
b_SELECTED=np.array([0.3805938025338775,0.4480306832558079,0.4044714907877873])
print('A=\n',A_SELECTED)
print('b=',b_SELECTED)
print('axis colors=\n',axis_colors(A_SELECTED,b_SELECTED))
print('J_axis_OKLab=',J_axis(pack(A_SELECTED,b_SELECTED)))
print('J_local_OKLab=',J_local(pack(A_SELECTED,b_SELECTED)))
# Reference values: J_axis ≈ 0.01988329, J_local ≈ 0.43000622
# Axis colors ≈
# x: (0.93573437, 0.23480299, 0.17940064)
# y: (0.25468629, 0.89022115, 0.32326642)
# z: (0.11219979, 0.30719132, 0.86387713)